**RNN**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

In [5]:
df=pd.read_csv('/content/IMDB Dataset.csv', engine='python')

In [6]:
df.isnull().sum()

,0
review,0
sentiment,0


In [7]:
df.drop_duplicates(inplace=True)

**TEXT PREPROCESSING**

In [8]:
#converting to lowercase
df['review']=df['review'].str.lower()

In [9]:
#removing the urls
import re
def remove_url(text):
  text=re.sub( r"http\S+", "" , text)
  return text
df['review']=df['review'].apply(remove_url)

In [10]:
#removing punctuations
def remove_punctutaions(text):
  text=re.sub(r"[^\w\s]", "", text)
  return text
df['review']=df['review'].apply(remove_punctutaions)

In [11]:
#removing html tags
def remove_htmltags(text):
  text=re.sub(r"<.*?>", "", text)
  return text
df['review']=df['review'].apply(remove_htmltags)

In [12]:
#removing stopwords
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [13]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [16]:
def remove_stopwords(text):
  stop_words=set(stopwords.words('english'))
  word_tokens=word_tokenize(text)
  for word in tokens:
    if word in stop_words:
      text=text.replace(word,"")
  return text
df['review']=df['review'].apply(remove_stopwords)

In [17]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmg ...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love the time of money is a vi...,positive


In [18]:
#stemming
from nltk.stem import PorterStemmer
ps=PorterStemmer()
def stem_words(text):
  word_tokens=word_tokenize(text)
  for word in word_tokens:
    text=text.replace(word, ps.stem(word))
  return text
df['review']=df['review'].apply(stem_words)

In [19]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,positive
1,a wonder littl product br br the filmg techniq...,positive
2,i thought thi wa a wonder way to spend time on...,positive
3,basic there a famili where a littl boy jake th...,negative
4,petter mattei love the time of money is a vis...,positive


In [20]:
#encoding
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['sentiment']=le.fit_transform(df['sentiment'])

In [21]:
y = df["sentiment"]
X = df["review"]

In [26]:
#vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer(max_features=5000)
X=vectorizer.fit_transform(df["review"])

**DATASET AND DATALOADERS**

In [28]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [29]:
import torch
from torch.utils.data import TensorDataset,DataLoader
X_train=X_train.toarray()
X_test=X_test.toarray()

In [30]:
train_set=TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
    )
test_set=TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [31]:
train_loader=DataLoader(train_set, batch_size=64, shuffle=True)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

**BUILDING RNN**

In [32]:
import torch.nn as nn
import torch.optim as optimizer

In [33]:
class RNN(nn.Module):
  def __init__(self,input_size,hidden_size=128,num_layers=1):
    super().__init__()
    self.hidden_size=hidden_size
    self.num_layers=num_layers
    self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)
    self.fc=nn.Linear(hidden_size,1)
  def forward(self,x):
    h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
    out,_=self.rnn(x,h0)
    out=self.fc(out[:,-1:])
    return out

In [35]:
input_size=X_train.shape[1]
model=RNN(input_size)
criterion=nn.BCELoss()
optimizer=optimizer.Adam(model.parameters())

**TRAINING THE RNN**

In [37]:
num_epocs=10
for epoch in range(num_epocs):
  model.train()
  for x,y in train_loader:
    optimizer.zero_grad()
    x=x.unsqueeze(1)
    outputs=model(x)
    outputs=torch.sigmoid(outputs.squeeze())
    loss=criterion(outputs,y)
    loss.backward()
    optimizer.step()
  print(f"Epoch {epoch+1}/{num_epocs}, Loss: {loss.item():.4f}")

Epoch 1/10, Loss: 0.0962
Epoch 2/10, Loss: 0.2595
Epoch 3/10, Loss: 0.2480
Epoch 4/10, Loss: 0.2904
Epoch 5/10, Loss: 0.3620
Epoch 6/10, Loss: 0.2440
Epoch 7/10, Loss: 0.3019
Epoch 8/10, Loss: 0.3987
Epoch 9/10, Loss: 0.1814
Epoch 10/10, Loss: 0.2370


In [40]:
model.eval()
with torch.no_grad():
  correct_vals=0
  total_vals=0
  for x,y in test_loader:
    x=x.unsqueeze(1)
    outputs=model(x)
    predicted=torch.round(torch.sigmoid(outputs.squeeze()))
    total_vals+=y.size(0)
    correct_vals+=(predicted==y).sum().item()
  accuracy=(correct_vals/total_vals)*100
  print(f"Accuracy on test set: {accuracy:.4f}")

Accuracy on test set: 87.8895


### **Overall Goal**
This notebook performs sentiment analysis on the IMDB movie review dataset. It involves loading the data, extensive text preprocessing, converting text to numerical features using TF-IDF, building a simple Recurrent Neural Network (RNN) with PyTorch, training the model, and finally evaluating its performance.

### **1. Data Loading and Initial Inspection**

- **`import pandas as pd`, etc.**: Imports necessary libraries for data manipulation, numerical operations, plotting, and deep learning (TensorFlow is imported but not used directly in the RNN part; PyTorch is used for the RNN).
- **`df=pd.read_csv('/content/IMDB Dataset.csv', engine='python')`**: Loads the IMDB movie review dataset from a CSV file into a pandas DataFrame. `engine='python'` is used to handle potential parsing issues in the CSV.
- **`df.isnull().sum()`**: Checks for any missing values in each column of the DataFrame. The output shows no missing values.
- **`df.drop_duplicates(inplace=True)`**: Removes any duplicate rows from the DataFrame to ensure data uniqueness, modifying the DataFrame in place.

### **2. Text Preprocessing**
This section cleans and standardizes the review text for better model performance.

- **`df['review']=df['review'].str.lower()`**: Converts all characters in the 'review' column to lowercase.
- **`remove_url(text)`**: Defines a function to remove URLs from the text using regular expressions (`re.sub( r"http\S+", "" , text)`).
- **`remove_punctutaions(text)`**: Defines a function to remove punctuation marks from the text using regular expressions (`re.sub(r"[^\w\s]", "", text)`).
- **`remove_htmltags(text)`**: Defines a function to remove HTML tags (like `<br />`) from the text using regular expressions (`re.sub(r"<.*?>", "", text)`).
- **`nltk.download('stopwords')`, `nltk.download('punkt')`**: Downloads necessary NLTK data for stopword removal and tokenization.
- **`remove_stopwords(text)`**: Defines a function to remove common English stopwords from the text. It tokenizes the text and then removes words present in the NLTK English stopwords set.
- **`stem_words(text)`**: Defines a function to apply stemming (reducing words to their base or root form) using NLTK's `PorterStemmer`. This helps in reducing the vocabulary size and treating different forms of a word as the same.
- **`df.head()`**: Displays the first few rows of the DataFrame to show the effect of the preprocessing steps on the 'review' column.

### **3. Encoding and Vectorization**

- **`from sklearn.preprocessing import LabelEncoder`**: Imports `LabelEncoder` to convert categorical labels into numerical format.
- **`le=LabelEncoder()`**: Initializes the LabelEncoder.
- **`df['sentiment']=le.fit_transform(df['sentiment'])`**: Transforms the 'sentiment' column (e.g., 'positive', 'negative') into numerical labels (e.g., 1, 0).
- **`y = df["sentiment"]`, `X = df["review"]`**: Separates the features (review text) and labels (sentiment).
- **`from sklearn.feature_extraction.text import TfidfVectorizer`**: Imports `TfidfVectorizer` to convert text data into numerical TF-IDF (Term Frequency-Inverse Document Frequency) features.
- **`vectorizer=TfidfVectorizer(max_features=5000)`**: Initializes the TF-IDF vectorizer, limiting the vocabulary to the 5000 most frequent words.
- **`X=vectorizer.fit_transform(df["review"])`**: Fits the vectorizer to the review text and transforms the text into a sparse matrix of TF-IDF features.

### **4. Dataset and DataLoaders**

- **`from sklearn.model_selection import train_test_split`**: Imports the function to split data.
- **`X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)`**: Splits the dataset into training (80%) and testing (20%) sets, ensuring reproducibility with `random_state=42`.
- **`import torch`, `from torch.utils.data import TensorDataset,DataLoader`**: Imports PyTorch libraries.
- **`X_train=X_train.toarray()`, `X_test=X_test.toarray()`**: Converts the sparse TF-IDF matrices to dense NumPy arrays, which are typically required by PyTorch for direct tensor conversion.
- **`train_set=TensorDataset(...)`, `test_set=TensorDataset(...)`**: Creates PyTorch `TensorDataset` objects for the training and testing data, converting NumPy arrays to PyTorch tensors. This bundles features and labels together.
- **`train_loader=DataLoader(...)`, `test_loader=DataLoader(...)`**: Creates PyTorch `DataLoader` objects. These iteratively load data in batches during training and evaluation, with `batch_size=64` and shuffling enabled for both for better generalization.

### **5. Building the RNN Model**

- **`import torch.nn as nn`, `import torch.optim as optimizer`**: Imports PyTorch modules for neural network layers and optimizers.
- **`class RNN(nn.Module):`**: Defines a custom RNN class that inherits from `nn.Module`.
  - **`__init__(self, input_size, hidden_size=128, num_layers=1)`**: The constructor initializes the RNN layers.
    - `self.hidden_size`, `self.num_layers`: Stores the hidden state size and number of recurrent layers.
    - `self.rnn=nn.RNN(...)`: Defines the core RNN layer. `input_size` is the dimension of input features (5000 for TF-IDF), `hidden_size` is the dimension of the hidden state, `num_layers` is the number of stacked RNNs, and `batch_first=True` means the input tensor has batch dimension first.
    - `self.fc=nn.Linear(hidden_size,1)`: Defines a fully connected linear layer that maps the RNN's output (last hidden state) to a single output neuron for binary classification.
  - **`forward(self, x)`**: Defines the forward pass of the network.
    - `h0=torch.zeros(...)`: Initializes the initial hidden state `h0` as zeros. Its shape is `(num_layers, batch_size, hidden_size)`.
    - `out,_=self.rnn(x,h0)`: Passes the input `x` and initial hidden state `h0` through the RNN. `out` contains the output for each time step, and `_` is the final hidden state.
    - `out=self.fc(out[:,-1:])`: Takes the output from the *last* time step (`out[:,-1:]`) and passes it through the fully connected layer. This is because for sequence classification, we usually care about the final state of the sequence.
    - `return out`: Returns the final output from the linear layer.
- **`input_size=X_train.shape[1]`**: Sets the `input_size` for the RNN to the number of features in the TF-IDF vector (5000).
- **`model=RNN(input_size)`**: Instantiates the RNN model.
- **`criterion=nn.BCELoss()`**: Defines the loss function as Binary Cross-Entropy Loss, suitable for binary classification tasks.
- **`optimizer=optimizer.Adam(model.parameters())`**: Initializes the Adam optimizer, which will update the model's weights during training.

### **6. Training the RNN**

- **`num_epocs=10`**: Sets the number of training epochs to 10.
- **`for epoch in range(num_epocs):`**: Loops through each epoch.
  - **`model.train()`**: Sets the model to training mode. This enables dropout (if any) and batch normalization (if any).
  - **`for x,y in train_loader:`**: Iterates through batches of training data from the `train_loader`.
    - **`optimizer.zero_grad()`**: Clears the gradients from the previous iteration.
    - **`x=x.unsqueeze(1)`**: Adds an extra dimension to the input `x`. The RNN expects input in the format `(batch_size, sequence_length, input_size)`. Since TF-IDF vectors are considered a single time step, `sequence_length` is 1.
    - **`outputs=model(x)`**: Performs a forward pass to get predictions from the model.
    - **`outputs=torch.sigmoid(outputs.squeeze())`**: Applies the sigmoid activation function to the raw outputs (logits) to get probabilities between 0 and 1, and then `squeeze()` removes any single-dimensional entries.
    - **`loss=criterion(outputs,y)`**: Calculates the Binary Cross-Entropy loss between the predicted probabilities and the true labels.
    - **`loss.backward()`**: Computes gradients of the loss with respect to model parameters.
    - **`optimizer.step()`**: Updates the model's weights using the calculated gradients and the Adam optimizer.
  - **`print(...)`**: Prints the loss for the current epoch.

### **7. Evaluating the RNN**

- **`model.eval()`**: Sets the model to evaluation mode. This disables dropout and batch normalization to ensure consistent behavior during inference.
- **`with torch.no_grad():`**: Disables gradient calculations. This is important during evaluation to save memory and computation, as we don't need to update weights.
  - **`correct_vals=0`, `total_vals=0`**: Initializes counters for correctly predicted samples and total samples.
  - **`for x,y in test_loader:`**: Iterates through batches of test data.
    - **`x=x.unsqueeze(1)`**: Reshapes input `x` for the RNN as done during training.
    - **`outputs=model(x)`**: Gets raw predictions from the model.
    - **`predicted=torch.round(torch.sigmoid(outputs.squeeze()))`**: Converts logits to probabilities using sigmoid, then rounds to 0 or 1 for binary predictions.
    - **`total_vals+=y.size(0)`**: Adds the batch size to the total samples count.
    - **`correct_vals+=(predicted==y).sum().item()`**: Compares predictions with true labels and sums up the correct ones.
  - **`accuracy=(correct_vals/total_vals)*100`**: Calculates the final accuracy as a percentage.
  - **`print(...)`**: Prints the accuracy of the model on the test set.